In [1]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

import boto3
import sys
import json
from datetime import datetime
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

s3_client=boto3.client('s3')
args = getResolvedOptions(sys.argv, ['JOB_NAME','data_flow_type'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

data_flow_type = args['data_flow_type']

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 2880
Session ID: 800757ae-57bd-48ce-a7d7-0a1c3927e379
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 800757ae-57bd-48ce-a7d7-0a1c3927e379 to get into ready status...
Session 800757ae-57bd-48ce-a7d7-0a1c3927e379 

In [1]:
# check to see whether the dq check is success
dq_check_status_files_list_response=s3_client.list_objects_v2(Bucket='amzn-s3-data-engg-project-raw-bucket',Prefix='lambda-DQ-check-status-indicator/')
if len(dq_check_status_files_list_response['Contents'])==1:
    raise Exception("The 0 byte file mentioning the DQ check is success is not present")

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: d1e5f27f-74ca-45ee-9e94-a4f5446ff03b
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session d1e5f27f-74ca-45ee-9e94-a4f5446ff03b to get into ready status...
Session d1e5f27f-74ca-45ee-9e94-a4f5446ff03b has been created.
NameError: name 's3_client' is not defined


In [4]:
#read input file from s3 bucket
path_except_filename="s3://amzn-s3-data-engg-project-staging-bucket/"

# logic to retrieve the latest file
response=s3_client.list_objects_v2(Bucket='amzn-s3-data-engg-project-staging-bucket',Prefix=data_flow_type+'/')

all_files=response['Contents']
latest_file=sorted(all_files, key=lambda x: x['LastModified'])[-1]['Key']

ip_path =path_except_filename+latest_file
ip_df=spark.read.csv(ip_path,header=True,inferSchema=True)

In [1]:
# transformations
# 1. extract only the colms that are needed
# transform_df=ip_df.select("player_name","runs")

# select_df=ip_df.select("user_id","age","age_group","gender","country","occupation","education_level","income_bracket",
#                           "relationship_status","primary_platform","num_platforms_used","daily_screen_time_minutes",
#                           "preferred_content_type","primary_device","usage_purpose","posts_per_week","likes_per_day",
#                           "comments_per_day","shares_per_week","followers_count","following_count",
#                           "has_purchased_via_social","follows_influencers","ad_click_frequency","uses_privacy_settings",
#                           "experienced_cyberbullying","reports_fake_news_frequency","self_reported_mental_health_effect",
#                           "sleep_hours_per_night","addiction_level_1_to_10","productivity_impact","platform_satisfaction",
#                           "account_created_date","is_verified_account","is_content_creator","uses_ai_features","daily_notifications",
#                           "checks_phone_first_morning","uses_screen_time_limits")

select_df=ip_df.select("match_id","date","team_1","team_2")




# 2. drop duplicate rows
# drop_duplicate_df=select_df.dropDuplicates()

# 3. filter the data

# filter_df=drop_duplicate_df.filter((drop_duplicate_df["age"] > 13) & (drop_duplicate_df["daily_screen_time_minutes"] > 60))

# 4. add a new column and then set the value in based on a ocndition
# add a column stating that he is sleep defective or not, if the person is not sleeping more than 8.

# withAddedColmn_df=filter_df.withColumn("sleep_deficit", when(col("sleep_hours_per_night") < 8, "Yes").otherwise("No"))

# 5. type cast some of the columns
# cast "likes_per_day","comments_per_day"
# cast "posts_per_week", "shares_per_week"
# cast "followers_count","following_count"

# cast_df=withAddedColumn_df.withColumn("likes_per_day", col("likes_per_day").cast("string"))
# cast_df=cast_df.withColumn("comments_per_day", col("comments_per_day").cast("string"))
# cast_df=cast_df.withColumn("posts_per_week", col("posts_per_week").cast("string"))
# cast_df=cast_df.withColumn("shares_per_week", col("shares_per_week").cast("string"))
# cast_df=cast_df.withColumn("followers_count", col("followers_count").cast("string"))
# cast_df=cast_df.withColumn("following_count", col("following_count").cast("string"))


# 6. combine two columns and have a single coulmn that has it as array.
# combine the above pairs to a particular column and remove those columns
# array_df=cast_df.withColumn("user_activity_metrics", array("posts_per_week", "likes_per_day", "comments_per_day", "shares_per_week"))\
#     .withColumn("user_network_metrics", array("followers_count", "following_count"))\
#     .drop("posts_per_week", "likes_per_day", "comments_per_day", "shares_per_week","followers_count","following_count")

# 7. rename a column
# rename the user_id to id
# rename_df = rename_df.withColumnRenamed("user_id", "id")

# 9. do a concadination operation.
# concat_df=rename_df.withColumn("occupation_education", concat("occupation", lit("_"), "education_level")).drop(
# "occupation", "education_level")

# 10. repartition
# concat_df.repartition(2).write.mode("overwrite").option("header", "true").csv("arn:aws:s3:::amzn-s3-data-engg-project-final-bucket")

# code for getting all the files before
existing_files = set()
response_before=s3_client.list_objects_v2(Bucket='amzn-s3-data-engg-project-final-bucket',Prefix='output/')
existing_files = {obj['Key'] for obj in response_before.get('Contents', []) if obj['Key'].endswith('.csv')}

select_df.repartition(2).write.mode("append").option("header", "true").csv("s3://amzn-s3-data-engg-project-final-bucket/output")

response = s3_client.list_objects_v2(Bucket='amzn-s3-data-engg-project-final-bucket',Prefix='output/')
all_files_after = {obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')}

new_files = all_files_after - existing_files
new_file_urls = [f"s3://amzn-s3-data-engg-project-final-bucket/{file}" for file in new_files]

manifest = {"entries": [{"url": file_url, "mandatory": True} for file_url in new_file_urls]}
run_timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
manifest_key = f"manifests/manifest-{run_timestamp}.json"
    
s3_client.put_object(
        Bucket='amzn-s3-data-engg-project-final-bucket',
        Key=manifest_key,
        Body=json.dumps(manifest, indent=2)
    )

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: e86dbb61-069d-4fbf-9f2e-0df981d488d1
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session e86dbb61-069d-4fbf-9f2e-0df981d488d1 to get into ready status...
Session e86dbb61-069d-4fbf-9f2e-0df981d488d1 has been created.
SyntaxError: invalid syntax (<stdin>, line 13)
